## 0. Installation

In [ ]:
%pip install numpy h5py

# I. Load the datasets from Zenodo

Open a **new terminal** (outside this notebook) and run from the root of the project:

**Option 1: one (or several) specific dimension and chi** (--chi accept 30 or 43 or all)

```bash
python src/load_data.py --dim 2 3 4 --chi 30
```

**Option 2: the full database** \
⚠️ may take 10/20 minutes.

```bash
python src/load_data.py --dim all --chi all
```

Files are saved in data/chi.../D.../, you can follow the progress from there. \
Once the download is finished, come back to this notebook and continue.

# II. Create a new dataset

A new dataset is generated from the downloaded ones by exploiting the symmetries of the distribution:

* random shifts
* random flips
* random axis swaps

When needed (see IV. Take Home), projection and tiling are applied first, taking a point cloud from shape `(N, D)` to `(N × 2^i, D − j)`.

In [ ]:
import numpy as np
from src.new_data import new_data
X = new_data(N=1024, D=2, chi=30, seed=42)
np.save("new_data/N1024_D2_chi30_seed42.npy", X)

to run new_data from the terminal insteed: 
```bash
python src/new_data.py --dim 2 --npoints 1024 --chi 30 --seed 42
```

# III. Analysis

This requires the `blue-sampler` package (%pip install blue-sampler).

In [ ]:
import blue_sampler as blue
from src.new_data import new_data

X = new_data(N=1024, D=2, chi=30, seed=42)
blue.plot(X)
blue.plot_structure_factor(X)

# IV. Take Home

Download the datasets (from the root of the repository), either manually from Zenodo (DOI [10.5281/zenodo.23244510](https://doi.org/10.5281/zenodo.23244510)) \
or by running, in the terminal:

```bash
# everything: D = 2 to 16, plus 23 and 32, both chi
python src/load_data.py --dim all --chi all   
# specific dimensions, specific chi (faster)
python src/load_data.py --dim 2 3 4 --chi 30  
```

Build a new randomised (N, D) point cloud from the downloaded ones:

- from Python:
```python
from src.new_data import new_data
X = new_data(N=1024, D=2, chi=30, seed=42, data_dir="data")  # (N, D) np.float32 array 
```
- or from the terminal:
```bash
python src/new_data.py --dim 2 --npoints 1024 --chi 30 --seed 42
```

**The chi parameter** (30 or 43) selects the dataset. It controls the spectral properties: chi = 30 (χ = 0.30) gives `S(k) <= 1e-10` \
on a smaller exclusion region, chi = 43 (χ = 0.43) gives `S(k) <= 1e-4` on a larger one.

new_data.py performs, in this order:

1. **Projection**: if D is not stored (17 <= D <= 22 or 24 <= D <= 31), a dataset with D' > D is used and projected.
2. **Tiling**: if N >= 2^18 (about 260k), the largest stored cloud (N = 2^17) is tiled, see [TILING.md](TILING.md).
3. **Randomisation** (with the given seed): random shift x_new = (x - s) mod 1, random flips x_new[:, d] = 1 - x_new[:, d], \
and random axis swaps.

**Supported:** any N power of 2 with `2^10 <= N <= 2^24`, any `2 <= D <= 32`, `chi = {30, 43}`.

There is no hard limit on tiling, but we recommend not going far beyond N = 2^24: \
stealthiness degrades proportionally to N_t / N.